# 03 — Runtime Signals and Graph Control

Multigen gives operators fine-grained control over a running workflow via **signal methods**:

| Signal | API | Effect |
|--------|-----|--------|
| Pause | `interrupt(id)` | Halt after the current node finishes |
| Add node | `inject_node(id, node)` | Append a new node mid-run |
| Jump | `jump_to(id, node_id)` | Push a node to the front of the queue |
| Skip | `skip_node(id, node_id)` | Mark a node to be silently bypassed |
| Continue | `resume(id)` | Unpause and continue execution |

## Pipeline for This Notebook

```
parse_resume → match_skills → evaluate_experience → aggregate_scores → guardrail
```

We will:
1. Launch this 5-node graph
2. After `parse_resume` completes — interrupt
3. Inject a new `bias_check` node
4. Jump directly to `aggregate_scores` (skipping some intermediate work)
5. Skip the `guardrail` node
6. Resume and observe the final state

In [ ]:
import time
import pprint

from multigen import SyncMultigenClient, InjectNodeRequest
from multigen.dsl import GraphBuilder

ORCHESTRATOR_URL = "http://localhost:8009"
client = SyncMultigenClient(base_url=ORCHESTRATOR_URL, timeout=60.0)
assert client.ping(), "Orchestrator not reachable."
print("Connected.")

## 1. Build the 5-Node Graph

In [ ]:
graph_def = (
    GraphBuilder()
    .node("parse_resume")
        .agent("ResumeParserAgent")
        .params(output_format="json")
        .timeout(30)
        .done()
    .node("match_skills")
        .agent("SkillMatcherAgent")
        .params(job_title="Data Scientist")
        .timeout(30)
        .done()
    .node("evaluate_experience")
        .agent("ExperienceEvaluatorAgent")
        .params(min_years=3)
        .timeout(30)
        .done()
    .node("aggregate_scores")
        .agent("ScoreAggregatorAgent")
        .params(weights={"skills": 0.6, "experience": 0.4})
        .timeout(20)
        .done()
    .node("guardrail")
        .agent("GuardrailAgent")
        .params(policy="hr_compliance_v2")
        .timeout(15)
        .done()
    .edge("parse_resume",       "match_skills")
    .edge("match_skills",       "evaluate_experience")
    .edge("evaluate_experience", "aggregate_scores")
    .edge("aggregate_scores",    "guardrail")
    .entry("parse_resume")
    .max_cycles(10)
    .build()
)

print("Nodes:", [n["id"] for n in graph_def["nodes"]])

## 2. Launch the Workflow

In [ ]:
payload = {
    "resume_text": "Alice Chen, 4 years ML engineering, PyTorch, scikit-learn, AWS SageMaker.",
    "candidate_id": "candidate-002",
}

response = client.run_graph(graph_def=graph_def, payload=payload)
instance_id = response.instance_id
print(f"Launched — instance_id: {instance_id}")

## 3. Wait for the First Node, Then Interrupt

We poll `get_node_state` for `parse_resume`. Once it has output, we immediately send an `interrupt` signal.

`interrupt()` tells the orchestrator to pause execution **after the current in-flight node completes**. No nodes are killed mid-execution.

In [ ]:
def wait_for_node(client, instance_id, node_id, max_wait=30):
    deadline = time.time() + max_wait
    while time.time() < deadline:
        try:
            ns = client.get_node_state(instance_id, node_id)
            if ns.output:
                return ns
        except Exception:
            pass
        time.sleep(1)
    return None

print("Waiting for 'parse_resume' to complete...")
ns = wait_for_node(client, instance_id, "parse_resume")
if ns:
    print(f"'parse_resume' completed at {ns.updated_at}")
else:
    print("Warning: node did not complete in time — interrupting anyway")

# Send interrupt
result = client.interrupt(instance_id)
print(f"\nInterrupt sent: {result}")

## 4. Verify Interrupted State

`get_health()` queries the **live** workflow (no DB round-trip) and returns:
- `interrupted` — whether the pause flag is set
- `pending_count` — how many nodes are queued but not yet executed
- `skip_nodes` — nodes marked for skipping
- `cb_trips_total` — cumulative circuit-breaker trips

In [ ]:
time.sleep(1)  # give the orchestrator a moment to register the interrupt
health = client.get_health(instance_id)

print("Workflow Health (after interrupt)")
print("-" * 35)
print(f"  interrupted   : {health.interrupted}")
print(f"  pending_count : {health.pending_count}")
print(f"  skip_nodes    : {health.skip_nodes}")
print(f"  cb_trips_total: {health.cb_trips_total}")
print(f"  errors        : {health.errors}")

## 5. Inject a New Node

`inject_node()` dynamically appends a new node to the live graph. The `edges_to` field specifies which existing node it should connect to as a predecessor.

Here we inject a `bias_check` node that runs `GuardrailAgent` with a bias-detection policy.

In [ ]:
injected_node = InjectNodeRequest(
    id="bias_check",
    agent="GuardrailAgent",
    params={"policy": "bias_detection_v1", "strict": False},
    edges_to=["aggregate_scores"],  # wire it before aggregate_scores
    timeout=20,
)

inject_result = client.inject_node(instance_id, injected_node)
print(f"Node injected: {inject_result}")

## 6. Jump to a Specific Node

`jump_to()` pushes `aggregate_scores` to the **front** of the execution queue. When the workflow resumes, this node will be executed immediately — regardless of normal traversal order.

This is useful for operator intervention: e.g., fast-pathing a candidate to scoring after a manual review.

In [ ]:
jump_result = client.jump_to(instance_id, "aggregate_scores")
print(f"Jump signal sent: {jump_result}")

## 7. Skip a Node

`skip_node()` marks `guardrail` to be silently bypassed. When the traversal reaches it, the node is logged as skipped and execution continues to the next node in the graph.

In [ ]:
skip_result = client.skip_node(instance_id, "guardrail")
print(f"Skip signal sent: {skip_result}")

# Verify the skip node is registered in health
health = client.get_health(instance_id)
print(f"skip_nodes after skip signal: {health.skip_nodes}")

## 8. Resume the Workflow

`resume()` clears the interrupt flag. Execution picks up from where it paused, applying the jump-to and skip signals we sent.

In [ ]:
resume_result = client.resume(instance_id)
print(f"Resume signal sent: {resume_result}")

# Give the workflow time to finish
time.sleep(5)

health = client.get_health(instance_id)
print(f"\nHealth after resume:")
print(f"  interrupted   : {health.interrupted}")
print(f"  pending_count : {health.pending_count}")

## 9. Final State

Inspect which nodes actually executed (note: `guardrail` should be absent or marked skipped in metrics).

In [ ]:
final_state = client.get_state(instance_id)
print(f"Final state — {final_state.count} node(s) completed:")
for ns in final_state.nodes:
    print(f"  {ns.node_id}: {list(ns.output.keys())}")

metrics = client.get_metrics(instance_id)
print(f"\nMetrics:")
print(f"  nodes_executed : {metrics.nodes_executed}")
print(f"  nodes_skipped  : {metrics.nodes_skipped}")

In [ ]:
client.close()
print("Done.")

---

## Summary of Signal Methods

```
interrupt(id)              → pause after current node
get_health(id)             → check live state (interrupted, pending, skip_nodes, errors)
inject_node(id, node_req)  → add a new node with optional edges
jump_to(id, node_id)       → execute this node next (priority)
skip_node(id, node_id)     → bypass a node silently
resume(id)                 → clear interrupt, continue from queue
```

**Next**: See `04_circuit_breakers.ipynb` for automatic failure handling.